[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uwarring82/iontrap-dynamics/blob/main/docs/tutorials/notebooks/16_two_mode_squeezing.ipynb)

**Run the _Setup_ cell below first**, then run the remaining cells top-to-bottom. The `assert` statements are the tutorial's built-in checks — if every cell runs without error, every step passed.

> _Auto-generated from [`docs/tutorials/16_two_mode_squeezing.md`](https://github.com/uwarring82/iontrap-dynamics/blob/main/docs/tutorials/16_two_mode_squeezing.md) by `tools/build_tutorial_notebooks.py`. Edit the Markdown tutorial, not this notebook._

In [ ]:
# Setup — install iontrap-dynamics and its dependencies (qutip, numpy, scipy).
# First run on Colab takes ~1-2 min; safe to re-run (a no-op once installed).
%pip install -q "iontrap-dynamics[plot] @ git+https://github.com/uwarring82/iontrap-dynamics.git@main"

# Tutorial 16 — Two-mode SU(1,1) squeezing


**Goal.** Couple two motional modes parametrically and grow entangled,
photon-number-correlated squeezing. By the end you will have built a
two-mode squeezed vacuum from the state factory, reproduced it dynamically
by evolving the two-mode vacuum under the SU(1,1) Hamiltonian, watched the
per-mode occupation grow as `sinh²(gτ)` while the difference number stays
pinned, and contrasted it against its SU(2) partner the beamsplitter.

**Reference implementation.** `tools/run_benchmark_two_mode_squeezing.py`,
with the committed plot under
[`benchmarks/data/two_mode_squeezing/`](https://github.com/uwarring82/iontrap-dynamics/tree/main/benchmarks/data/two_mode_squeezing).

**Expected time.** ~13 min reading; ~2 s runtime.

**Level.** `advanced` — a specialised or research-grade surface; do the core first.

**Prerequisites.** [Tutorial 9](https://uwarring82.github.io/iontrap-dynamics/tutorials/09_squeezed_coherent_prep/) (single-mode
squeezed/coherent factories) and [Tutorial 6](https://uwarring82.github.io/iontrap-dynamics/tutorials/06_fock_truncation/)
(Fock-truncation diagnosis — you will need it here). CONVENTIONS.md §23
fixes the two-mode squeezing phase/sign and the `n̄ = sinh²|z|` occupation
convention. This is the first tutorial to use a Hilbert space with **two**
motional modes.

---

> 📝 **Note** — New here? Read this first
>
>
> - A **two-mode squeezer** creates and destroys quanta in *pairs* — one in mode `a` and one in mode `b` at a time — so both modes' occupation grows *together*, never one without the other.
> - The growth is exponential (parametric gain): per-mode `⟨n̂⟩ = sinh²|z|` for the static factory state, and `sinh²(gτ)` as you evolve the vacuum for a time `τ`.
> - Because quanta arrive in balanced pairs, the **difference** `n̂_a − n̂_b` is conserved — it stays `0` from the vacuum — the SU(1,1) Casimir. That perfect number correlation is the entanglement resource.
> - Either mode *on its own* is just a thermal state with that `n̄`; the squeezing lives only in the `a`–`b` correlation, invisible mode-by-mode.
> - The **beamsplitter** (Step 3) is the opposite generator: SU(2) rotates one quantum between the modes and conserves the **sum** `n̂_a + n̂_b` — do not conflate the two conservation laws.
> - Both modes climb the Fock ladder fast, so size the per-mode truncation generously ([Tutorial 6](https://uwarring82.github.io/iontrap-dynamics/tutorials/06_fock_truncation/)) or the occupation readout is biased.
>
> **In a hurry?** Step 1 *declares* the state from the factory; Step 2 *rebuilds* it dynamically and shows the `sinh²(gτ)` growth with the difference pinned — that pair is the core. Step 3 contrasts the beamsplitter.

**Symbols in this tutorial**

| symbol | plain meaning |
| --- | --- |
| `z` | two-mode squeeze parameter (real here, so it equals `r`); a complex phase would set the squeezing axis `φ` |
| `r` | magnitude of the squeeze parameter — set directly as the factory `z` in Step 1, built up as `r = gτ` (coupling `g`, time `τ`) in Step 2 |
| `n̄ = sinh²(r)` | per-mode mean phonon number — grows exponentially in `r` |
| `g` | parametric coupling rate of the two-mode squeezing Hamiltonian |
| `n̂_a − n̂_b` | difference number — the SU(1,1) Casimir; conserved, stays `0` from the vacuum |
| `n̂_a + n̂_b` | sum number — conserved instead by the SU(2) beamsplitter |
| `J` | beamsplitter coupling rate (Step 3); a full a→b swap takes `π/(2J)` (`n_a` oscillates with period `π/J`) |

## The scenario

A single-mode squeezer reshapes one oscillator's quadratures. A *two-mode*
squeezer, generated by `H/ℏ = i g (e^{iφ} â†b̂† − e^{−iφ} âb̂)`, instead
creates and destroys photons **in pairs**, one in each mode. The result is
SU(1,1) physics: the total photon number is *unbounded* (it grows
exponentially — parametric amplification), but the **difference**
`n̂_a − n̂_b` is conserved. Starting from the vacuum the two modes stay
perfectly photon-number correlated — the defining property of the two-mode
squeezed vacuum, and the resource behind SU(1,1) interferometry and
EPR-type entanglement.

We will meet this state twice: as a *static* object from the factory, and
as the *dynamical* endpoint of evolving the vacuum under the squeezing
Hamiltonian. They must agree.

## Step 1 — The two-mode squeezed-vacuum factory

`two_mode_squeezed_vacuum(fock_dim, z)` builds the state directly. Its
per-mode occupation is `⟨n̂⟩ = sinh²|z|` (CONVENTIONS §23 — note this is
**not** qutip's half-angle convention), and its Fock-grid amplitudes are
diagonal: only `|n, n⟩` terms appear, the photon-number correlation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qutip
from iontrap_dynamics.states import two_mode_squeezed_vacuum

# House colours — match the reference figure style.
BLUE, RED, GREEN, PURPLE, GREY = "#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#444444"

fock = 40
factory_r = [0.3, 0.6, 1.0]
factory_nbar = []
for r in factory_r:
    tmsv = two_mode_squeezed_vacuum(fock, r)
    n_a = qutip.tensor(qutip.num(fock), qutip.qeye(fock))
    nbar = float(qutip.expect(n_a, tmsv))
    factory_nbar.append(nbar)
    print(f"Step 1 — r = {r:.1f}:  ⟨n̂⟩ (factory) = {nbar:.4f},  sinh²(r) = {np.sinh(r)**2:.4f}")
    assert abs(nbar - np.sinh(r) ** 2) < 1e-3, "factory ⟨n̂⟩ must equal sinh²|z| — the §23 two-mode occupation convention"   # ⟨n̂⟩ = sinh²|z|
# the reduced single-mode state is thermal with this n̄ — squeezing you can only see in the pair

> 📝 **Note** — The convention that bites
>
>
> Two-mode `⟨n̂⟩ = sinh²|z|`, not `sinh²(|z|/2)`. CONVENTIONS §23 fixes
> this explicitly because `qutip.squeezing` uses the half-angle form;
> feeding `qutip`'s `z` into a `sinh²|z|` formula (or vice versa) is the
> classic factor-of-two error. `z` may be complex — its phase is the
> squeezing axis `φ`; here we take it real.

## Step 2 — The same state, dynamically

Now build the state instead of declaring it. Evolve the two-mode vacuum
under `two_mode_squeezing_hamiltonian` for a time `τ`: the squeezing grows
as `r = gτ`, so the per-mode occupation should trace `sinh²(gτ)` and the
difference number `⟨n̂_a − n̂_b⟩` should never leave zero.

In [ ]:
from iontrap_dynamics.hilbert import HilbertSpace
from iontrap_dynamics.system import IonSystem
from iontrap_dynamics.species import mg25_plus
from iontrap_dynamics.modes import ModeConfig
from iontrap_dynamics.hamiltonians import two_mode_squeezing_hamiltonian
from iontrap_dynamics.observables import Observable
from iontrap_dynamics.sequences import solve

# Plumbing — build a two-mode Hilbert space: spin ⊗ mode a ⊗ mode b.
def two_mode_hilbert(fock_dim):
    ev = np.array([[0.0, 0.0, 1.0]])
    modes = (ModeConfig(label="a", frequency_rad_s=2 * np.pi * 1.0e6, eigenvector_per_ion=ev),
             ModeConfig(label="b", frequency_rad_s=2 * np.pi * 1.1e6, eigenvector_per_ion=ev))
    system = IonSystem(species_per_ion=(mg25_plus(),), modes=modes)
    return HilbertSpace(system=system, fock_truncations={"a": fock_dim, "b": fock_dim})

# The physics — evolve the two-mode vacuum under the squeezing H for a growing time τ (r = g·τ).
FOCK = 50                                       # generous: n̄ reaches ≈ 2.3 at r = 1.2
g = 2 * np.pi * 2.0e3
h = two_mode_hilbert(FOCK)
H = two_mode_squeezing_hamiltonian(h, g=g)
vacuum = qutip.tensor(qutip.basis(2, 0), qutip.basis(FOCK, 0), qutip.basis(FOCK, 0))
times = np.linspace(0.0, 1.2 / g, 40)           # r = g·τ up to 1.2
res = solve(hilbert=h, hamiltonian=H, initial_state=vacuum, times=times,
            observables=(Observable(label="n_a", operator=h.number_for_mode("a")),
                         Observable(label="n_b", operator=h.number_for_mode("b"))))

n_a = np.asarray(res.expectations["n_a"]); n_b = np.asarray(res.expectations["n_b"])
r_t = g * times
nbar_analytic = np.sinh(r_t) ** 2
max_err_occ = float(np.max(np.abs(n_a - nbar_analytic)))
max_diff = float(np.max(np.abs(n_a - n_b)))
print(f"Step 2 — max |⟨n̂_a⟩ − sinh²(gτ)| = {max_err_occ:.2e}  (oracle < 1e-4)")
print(f"Step 2 — max |⟨n̂_a⟩ − ⟨n̂_b⟩|    = {max_diff:.2e}  (oracle < 1e-9, SU(1,1) Casimir)")
print(f"Step 2 — final n̄ at r = {r_t[-1]:.2f}:  ⟨n̂_a⟩ = {n_a[-1]:.4f},  sinh²(r) = {nbar_analytic[-1]:.4f}")
assert np.max(np.abs(n_a - nbar_analytic)) < 1e-4, "evolved occupation must trace the analytic sinh²(gτ) curve"   # ⟨n̂_a⟩ = sinh²(gτ)
assert np.max(np.abs(n_a - n_b)) < 1e-9, "⟨n̂_a⟩ and ⟨n̂_b⟩ must stay equal — pairs are created one per mode, so the difference is conserved"             # difference number ≡ 0 (the su(1,1) Casimir)

# Plot 1 — parametric growth: dynamics, factory anchors, analytic curve.
fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.2))
ax = axes[0]
ax.plot(r_t, nbar_analytic, color=GREY, linewidth=1.0, label=r"$\sinh^2(g\tau)$")
ax.plot(r_t, n_a, color=BLUE, marker="o", markersize=3, label=r"$\langle\hat{n}_a\rangle$ (dynamics)")
ax.plot(r_t, n_b, color=RED, marker="s", markersize=3, linestyle="--", label=r"$\langle\hat{n}_b\rangle$ (dynamics)")
ax.scatter(factory_r, factory_nbar, color=GREEN, zorder=5, s=40, marker="^", label="factory anchors")
ax.set_xlabel(r"squeezing parameter $r = g\tau$")
ax.set_ylabel(r"mean phonon number $\langle\hat{n}\rangle$")
ax.set_title("Two-mode squeezing: parametric growth")
ax.legend(frameon=False)
# Plot 2 — conserved difference number.
ax2 = axes[1]
ax2.plot(r_t, n_a - n_b, color=PURPLE, marker="o", markersize=3)
ax2.set_xlabel(r"squeezing parameter $r = g\tau$")
ax2.set_ylabel(r"$\langle\hat{n}_a - \hat{n}_b\rangle$")
ax2.set_title(r"SU(1,1) Casimir: difference number $\equiv 0$")
plt.tight_layout()
plt.show()

![Per-mode occupation growing as sinh squared of the squeezing while the difference number stays flat at zero](https://raw.githubusercontent.com/uwarring82/iontrap-dynamics/main/benchmarks/data/two_mode_squeezing/plot.png)

The left panel overlays the dynamics, the factory occupations, and the
analytic `sinh²(r)` curve — all on top of one another, the exponential
parametric gain. The right panel is the conserved difference number, flat
at zero across the whole evolution: every photon created in mode `a` has a
partner in mode `b`.

**Takeaway.** The green factory anchors landing exactly on the blue and red
dynamical curves is this tutorial's whole claim made visible — the *declared*
state (Step 1) and the *evolved* state (Step 2) are one and the same object,
and the flat right panel is that state's `|n, n⟩`-only amplitude structure
seen unfolding in real time.

> ⚠️ **Warning** — Fock truncation is the trap here
>
>
> `n̄ = sinh²(r)` blows up fast — `r = 1.5` already means `n̄ ≈ 4.6`, with
> population reaching well past `n = 10`. A too-small `fock_dim` makes
> `solve` emit a `FockQualityWarning` or raise `ConvergenceError` (the
> §15 saturation ladder from [Tutorial 6](https://uwarring82.github.io/iontrap-dynamics/tutorials/06_fock_truncation/)). Keep
> `r ≲ 1.3` with `fock_dim ≥ 48`, or raise the cutoff — the squeezed
> vacuum needs headroom in *both* modes.

## Step 3 — The SU(2) partner: the beamsplitter

The two-mode squeezer's algebraic sibling is the **beamsplitter**,
`beamsplitter_hamiltonian`, which generates SU(2) rather than SU(1,1). It
*rotates* excitation between the modes instead of amplifying it: the total
`n̂_a + n̂_b` is conserved while the difference oscillates — the exact
opposite conservation law. Start with a single phonon in mode `a` and watch
it slosh into mode `b` and back.

> ⚠️ **Warning** — Common confusion — opposite conservation laws
>
>
> Do not swap the two conservation laws. The **SU(1,1) squeezer**
> (`â†b̂† + âb̂`) creates quanta in balanced pairs, so it conserves the
> **difference** `n̂_a − n̂_b` while the **sum** grows without bound. The
> **SU(2) beamsplitter** (`â†b̂ + âb̂†`) only relocates existing quanta, so it
> conserves the **sum** `n̂_a + n̂_b` while the difference oscillates.
> Growth-with-correlation versus mixing-at-fixed-total: one generator
> amplifies, the other rotates.

In [ ]:
from iontrap_dynamics.hamiltonians import beamsplitter_hamiltonian

J = 2 * np.pi * 2.0e3
H_bs = beamsplitter_hamiltonian(h, coupling=J)
one_a = qutip.tensor(qutip.basis(2, 0), qutip.basis(FOCK, 1), qutip.basis(FOCK, 0))  # |1⟩_a |0⟩_b
times_bs = np.linspace(0.0, np.pi / (2 * J), 30)
res_bs = solve(hilbert=h, hamiltonian=H_bs, initial_state=one_a, times=times_bs,
               observables=(Observable(label="n_a", operator=h.number_for_mode("a")),
                            Observable(label="n_b", operator=h.number_for_mode("b"))))
na = np.asarray(res_bs.expectations["n_a"]); nb = np.asarray(res_bs.expectations["n_b"])
max_sum_err = float(np.max(np.abs((na + nb) - 1.0)))
print(f"Step 3 — max |⟨n̂_a + n̂_b⟩ − 1| = {max_sum_err:.2e}  (oracle < 1e-6, SU(2) total conserved)")
print(f"Step 3 — at t = π/(2J):  ⟨n̂_a⟩ = {na[-1]:.2e} (→ 0),  ⟨n̂_b⟩ = {nb[-1]:.4f} (→ 1)  [full swap]")
assert np.max(np.abs((na + nb) - 1.0)) < 1e-6, "beamsplitter must conserve the total ⟨n̂_a + n̂_b⟩ = 1 — it moves the one phonon, never makes a pair"           # n̂_a + n̂_b conserved (SU(2))
assert abs(na[-1]) < 1e-6 and abs(nb[-1] - 1.0) < 1e-6  # a full swap a → b at t = π/(2J)

# Plot — SU(2) beamsplitter: phonon hopping between the two modes.
tau_bs = times_bs * J / np.pi  # phase in units of π/(2J) swap period
analytic_na = np.cos(J * times_bs) ** 2
analytic_nb = np.sin(J * times_bs) ** 2
fig, ax = plt.subplots(figsize=(5.0, 3.2))
ax.plot(tau_bs, analytic_na, color=GREY, linewidth=1.0, label=r"$\cos^2(J t)$")
ax.plot(tau_bs, analytic_nb, color=GREY, linewidth=1.0, linestyle="--")
ax.scatter(tau_bs, na, color=BLUE, s=14, zorder=3, label=r"$\langle\hat{n}_a\rangle$")
ax.scatter(tau_bs, nb, color=RED, marker="s", s=14, zorder=3, label=r"$\langle\hat{n}_b\rangle$")
ax.set_xlabel(r"normalised time $J t / \pi$")
ax.set_ylabel(r"phonon number")
ax.set_title(r"SU(2) beamsplitter: $|1\rangle_a|0\rangle_b \to |0\rangle_a|1\rangle_b$")
ax.legend(frameon=False)
plt.show()

**Takeaway.** The seed matters: from the *vacuum* the beamsplitter would sit
frozen — `â†b̂ + âb̂†` annihilates `|0, 0⟩`, so with nothing to move it does
nothing — which is exactly why Step 3 starts from `|1⟩_a`, whereas the
squeezer makes its own pairs and could start from the vacuum.

The contrast is the whole point: the **squeezer** conserves the difference
and grows the sum (entanglement generation); the **beamsplitter** conserves
the sum and moves the difference (mode mixing). Both builders are
label-based and order-agnostic — they act on the modes you name in
`mode_labels`, regardless of tensor layout.

## Where to next

- [Tutorial 17 — Motional decoherence and the Lamb–Dicke regime](https://uwarring82.github.io/iontrap-dynamics/tutorials/17_motional_decoherence_and_lamb_dicke/):
  reuses this two-mode/single-mode `ModeConfig` setup, then lets the motion
  *decohere* under typed channels — and quantifies how badly.
- The benchmark `tools/run_benchmark_two_mode_squeezing.py` reproduces the
  figure and adds the static-factory cross-check across a sweep of `|z|`.

---

## Licence

Sail material — adaptive guidance with specific parameter choices, not a
coastline constraint. Licensed under **CC BY-NC-SA 4.0** per
[`docs/LICENCE`](https://github.com/uwarring82/iontrap-dynamics/blob/main/docs/LICENCE).